# Proč projekty nabírají skluz

*Nad rámec skripta — aplikace.*

---

Každý úkol v projektu má odhad trvání. Odhady jsou poctivé — v průměru sedí, občas se to stihne dřív, občas později. Přesto projekt skoro nikdy nestihne termín spočítaný z průměrů.

Není to nekompetencí ani pesimismem. Je to **matematika**:

$$E\big[\max(X_1, \dots, X_n)\big] \;>\; \max\big(E X_1, \dots, E X_n\big).$$

Projekt čeká na **nejpomalejší** větev, a maximum náhodných veličin je systematicky větší než maximum jejich průměrů. Čím víc souběžných větví, tím horší.

Tahle úloha má v jednom případě **přesné řešení**, které to ukáže jako na dlani: pro $n$ nezávislých úkolů s rozdělením $\operatorname{Exp}(1)$ je

$$E\big[\max\big] = H_n = 1 + \tfrac12 + \tfrac13 + \dots + \tfrac1n.$$

Každý úkol trvá v průměru 1. Deset souběžných úkolů trvá v průměru **2.93**.

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def estimate(xs):
    n = len(xs)
    m = sum(xs) / n
    var = sum((x - m) ** 2 for x in xs) / (n - 1)
    return m, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    print(f"{name}: {mean:.4f} +- {se:.4f}  (přesně {exact:.4f})")
    assert abs(mean - exact) <= k * se, name


def harmonicke(n):
    return sum(1.0 / k for k in range(1, n + 1))

## 1. Čekání na nejpomalejšího

Nejjednodušší možný projekt: $n$ úkolů běží **souběžně** a projekt končí, až doběhne poslední. Každý úkol trvá $\operatorname{Exp}(1)$, tedy v průměru jednotku času.

Naivní odhad říká: všechny trvají v průměru 1, takže projekt trvá 1. Skutečnost je $H_n$.

In [ ]:
n_vzorku = 200_000
print(f"{'úkolů':>7s} {'naivní odhad':>14s} {'skutečnost':>22s} {'přesně $H_n$':>14s} {'skluz':>8s}")
pocty = (1, 2, 3, 5, 10, 20, 50)
skutecne = []
for n in pocty:
    vzorky = [max(random.expovariate(1.0) for _ in range(n)) for _ in range(n_vzorku // max(1, n // 4))]
    m, se = estimate(vzorky)
    skutecne.append(m)
    print(f"{n:>7d} {1.0:>14.2f} {m:>14.4f} +- {se:.4f} {harmonicke(n):>14.4f} {m / 1.0:>7.1f}x")
    assert abs(m - harmonicke(n)) < 5 * se

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax = axes[0]
nn = list(range(1, 51))
ax.plot(nn, [harmonicke(n) for n in nn], "r-", lw=2, label="$E[\\max] = H_n$ (přesně)")
ax.plot(pocty, skutecne, "o", ms=7, label="simulace")
ax.axhline(1.0, color="k", ls="--", lw=1, label="naivní odhad (průměr úkolu)")
ax.set_xlabel("počet souběžných úkolů $n$")
ax.set_ylabel("očekávaná doba projektu")
ax.set_title("Čekání na nejpomalejšího roste jako $\\ln n$", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
for n, barva in ((1, "tab:blue"), (5, "tab:orange"), (20, "tab:red")):
    vz = [max(random.expovariate(1.0) for _ in range(n)) for _ in range(40_000)]
    w = 8 / 60
    ax.hist(vz, bins=60, range=(0, 8), weights=[1 / (len(vz) * w)] * len(vz),
            histtype="step", lw=1.5, color=barva, label=f"$n$ = {n}")
ax.axvline(1.0, color="k", ls="--", lw=1)
ax.set_xlabel("doba projektu")
ax.set_ylabel("hustota")
ax.set_title("Rozdělení se celé posouvá doprava", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Dvacet souběžných úkolů, každý v průměru na jednotku času, zabere v průměru **3.6** jednotky. Žádný úkol se nezpozdil — jen se čeká na toho, komu se zrovna nedařilo.

Poučení pro plánování: **paralelizace nepomáhá tak, jak se zdá.** Rozdělit práci na dvacet souběžných větví nezkrátí dobu na dvacetinu, protože výsledek řídí nejpomalejší větev, a ta je tím pomalejší, čím víc větví máte.

## 2. Skutečný projekt: síť úkolů

V praxi úkoly nejsou všechny souběžné — jsou propojené závislostmi. Projekt se modeluje jako **orientovaný graf**, kde hrana znamená „tohle musí skončit, než tamto začne". Doba projektu je **nejdelší cesta** sítí (kritická cesta).

Vezměme malý projekt o osmi úkolech. Doby trvání modelujeme trojúhelníkovým rozdělením (cvičení 4) — optimistický, nejpravděpodobnější a pesimistický odhad, přesně jak to dělá klasická metoda PERT.

In [ ]:
def triangularni(a, c, b):
    """Trojúhelníkové rozdělení na [a, b] s vrcholem v c (cvičení 4)."""
    g = random.random()
    hranice = (c - a) / (b - a)
    if g < hranice:
        return a + math.sqrt(g * (b - a) * (c - a))
    return b - math.sqrt((1.0 - g) * (b - a) * (b - c))


# úkol: (název, předchůdci, optimisticky, nejpravděpodobněji, pesimisticky)
PROJEKT = [
    ("analýza",      [],                        3,  5, 12),
    ("návrh UI",     ["analýza"],               2,  4,  9),
    ("návrh API",    ["analýza"],               3,  5,  8),
    ("frontend",     ["návrh UI"],              5,  8, 20),
    ("backend",      ["návrh API"],             6, 10, 18),
    ("databáze",     ["návrh API"],             2,  4, 11),
    ("integrace",    ["frontend", "backend", "databáze"], 2, 4, 10),
    ("testování",    ["integrace"],             3,  6, 15),
]
NAZVY = [u[0] for u in PROJEKT]


def doba_projektu(vraceni_kriticke=False):
    """Nejdelší cesta sítí. Vrací dobu, případně i který úkol byl poslední."""
    konec = {}
    for nazev, pred, a, c, b in PROJEKT:
        zacatek = max((konec[p] for p in pred), default=0.0)
        konec[nazev] = zacatek + triangularni(a, c, b)
    posledni = max(konec, key=konec.get)
    return (konec[posledni], posledni) if vraceni_kriticke else konec[posledni]


# naivní odhad: všechny úkoly na své střední hodnotě
def naivni():
    konec = {}
    for nazev, pred, a, c, b in PROJEKT:
        zacatek = max((konec[p] for p in pred), default=0.0)
        konec[nazev] = zacatek + (a + c + b) / 3
    return max(konec.values())


N = 100_000
doby = [doba_projektu() for _ in range(N)]
m, se = estimate(doby)
doby_s = sorted(doby)
print(f"naivní plán (každý úkol na průměru): {naivni():.2f} dní")
print(f"simulace, střední doba:              {m:.2f} +- {se:.2f} dní")
print(f"                                     tedy o {m / naivni() - 1:.0%} déle\n")
for q in (0.5, 0.8, 0.9, 0.95, 0.99):
    print(f"  termín, který stihneme s pravděpodobností {q:4.0%}: {doby_s[int(q * N)]:.1f} dní")
print(f"\nNaivní plán stihneme jen v {sum(1 for d in doby if d <= naivni()) / N:.0%} případů.")

Tohle je jádro věci. Plán spočítaný z průměrů vyjde optimisticky a **stihne se jen asi ve třetině případů** — přestože každý jednotlivý odhad byl poctivý.

Chcete-li termín, který se dá slíbit, musíte vzít **kvantil**, ne průměr. Simulace ho dá rovnou, stejně jako u potřebného kapitálu pojišťovny ve cvičení 9.

## 3. Kritická cesta není pevná

Klasické plánování určí jednu „kritickou cestu" a tu hlídá. Jenže která větev se stane kritickou, je samo náhodné — při každém běhu může skluz způsobit jiný úkol.

Spočítáme, **jak často je který úkol tím posledním**, a jak moc záleží na jeho zdržení.

In [ ]:
from collections import Counter

PREDCHUDCI = {nazev: pred for nazev, pred, *_ in PROJEKT}


def kriticka_cesta():
    """Jeden běh; vrací (doba projektu, množina úkolů na kritické cestě).

    Kritickou cestu najdeme zpětně: od posledního úkolu vždy skočíme na toho
    předchůdce, který skončil nejpozději — právě on určil, kdy se mohlo začít.
    """
    konec = {}
    for nazev, pred, a, c, b in PROJEKT:
        zacatek = max((konec[p] for p in pred), default=0.0)
        konec[nazev] = zacatek + triangularni(a, c, b)
    uzel = max(konec, key=konec.get)
    cesta = set()
    while uzel is not None:
        cesta.add(uzel)
        pred = PREDCHUDCI[uzel]
        uzel = max(pred, key=konec.get) if pred else None
    return konec[max(konec, key=konec.get)], cesta


na_ceste = Counter()
M_ = 60_000
for _ in range(M_):
    _, cesta = kriticka_cesta()
    na_ceste.update(cesta)

print("Jak často leží úkol na kritické cestě:")
for nazev, pocet in na_ceste.most_common():
    print(f"  {nazev:>12s} {pocet / M_:>7.1%}")
print("\nÚkoly se 100 % leží na kritické cestě vždy (nemají souběžnou alternativu).")
print("Zajímavé jsou ty mezi: které z větví frontend / backend / databáze zrovna brzdí,")
print("se mění od běhu k běhu — a právě proto nelze hlídat jen jednu pevnou cestu.")

# citlivost: o kolik se zkrátí projekt, když jeden úkol zrychlíme o 20 %
zaklad = sum(doba_projektu() for _ in range(40_000)) / 40_000
print(f"\nzákladní střední doba: {zaklad:.2f} dní")
print("\nCo získáme, když JEDEN úkol zrychlíme o 20 %:")
zisky = {}
for i, (nazev, pred, a, c, b) in enumerate(PROJEKT):
    puvodni = PROJEKT[i]
    PROJEKT[i] = (nazev, pred, a * 0.8, c * 0.8, b * 0.8)
    nova = sum(doba_projektu() for _ in range(40_000)) / 40_000
    PROJEKT[i] = puvodni
    zisky[nazev] = zaklad - nova
for nazev, z in sorted(zisky.items(), key=lambda kv: -kv[1]):
    print(f"  {nazev:>12s} {z:>6.2f} dní")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
ax = axes[0]
jm = [n for n, _ in na_ceste.most_common()]
ax.barh(jm, [na_ceste[n] / M_ for n in jm], color="tab:blue", alpha=0.75)
ax.set_xlabel("podíl běhů, kdy úkol leží na kritické cestě")
ax.set_title("Kritická cesta se mění od běhu k běhu", fontsize=10)
ax.invert_yaxis()

ax = axes[1]
jm2 = sorted(zisky, key=lambda n: zisky[n])
b = ax.barh(jm2, [zisky[n] for n in jm2], color="tab:green", alpha=0.75)
ax.bar_label(b, fmt="%.2f", padding=3, fontsize=8)
ax.set_xlabel("zkrácení projektu [dny], když úkol zrychlíme o 20 %")
ax.set_xlim(right=max(zisky.values()) * 1.25)
ax.set_title("Kde se vyplatí zrychlovat", fontsize=10)
plt.tight_layout()
plt.show()

Pravý graf je praktická odpověď na otázku „kam dát lidi navíc". Zrychlit úkol, který skoro nikdy není poslední, projektu nepomůže skoro vůbec — zatímco stejná investice do úzkého hrdla se projeví přímo.

A levý graf vysvětluje, proč klasická kritická cesta klame. Úkoly bez souběžné alternativy (analýza, integrace, testování) leží na kritické cestě vždy — ty jsou nudné. Zajímavá je trojice **frontend / backend / databáze**: každá z nich se kritickou stane jen někdy, podle toho, které se zrovna nedaří. **Žádná jediná cesta není kritická pořád**, takže hlídat jen tu nominálně nejdelší znamená přehlížet větve, které se kritickými stanou při nepříznivém běhu.

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Jádro problému | $E[\max] > \max E$ — projekt čeká na nejpomalejší větev |
| Přesná kontrola | $n$ úkolů $\operatorname{Exp}(1)$ souběžně ⟹ $E[\max] = H_n$ |
| Paralelizace | dvacet větví po jedné jednotce trvá $3.6$, ne $1$ |
| Plán z průměrů | systematicky optimistický; stihne se jen asi ve třetině případů |
| Slíbitelný termín | **kvantil**, ne průměr — stejně jako kapitál pojišťovny |
| Kritická cesta | není pevná, mění se od běhu k běhu |
| Kam investovat | tam, kde je úkol často poslední, ne tam, kde je nominálně nejdelší |

Souvislost se zbytkem repozitáře: je to Jensenova nerovnost v praxi. Ve `vetvici_proces` rozhodovala zpětná vazba, tady rozhoduje **maximum** — a obojí je případ, kdy střední hodnota výsledku není výsledkem středních hodnot.

### Zkuste si

1. Ověřte $E[\max] = H_n$ i pro úkoly s jiným rozdělením se střední hodnotou 1 (třeba rovnoměrným na $[0,2]$). Platí vzorec? Proč ne?
2. Přidejte do projektu **závislost mezi úkoly** (když se zdrží analýza, zdrží se všechno). Jak se změní rozptyl doby projektu?
3. Spočítejte, o kolik se termín na 95 % posune, když zdvojnásobíte pesimistické odhady, a porovnejte s posunem průměru.
4. Zkuste „studentský syndrom": úkol nikdy neskončí dřív, než je naplánováno (useknutá levá část rozdělení). O kolik se projekt prodlouží?
5. Rozdělte velký úkol na dva souběžné poloviční. Zkrátí se projekt na polovinu? (Nápověda: podívejte se na $H_2$.)
6. Místo trojúhelníkového rozdělení použijte lognormální se stejnou střední hodnotou a rozptylem. Jak moc záleží na tvaru chvostu?